# Orbit Wars Pro - Kaggle Competition Solution
## Multi-Strategy AI Agent for Planet Wars Style Game

**Competition**: [Orbit Wars](https://www.kaggle.com/competitions/orbit-wars/)
**Prize Pool**: $50,000 | **Deadline**: June 23, 2026

### Features:
- **5 Strategy Modes**: Adaptive, Greedy, Aggressive, Defensive, Hybrid
- **Game State Analysis**: Phase detection, threat assessment, target scoring
- **Combat Optimization**: Fleet size calculation, risk evaluation
- **Pathfinding**: Sun collision avoidance, trajectory planning

**Strategy Selection by Game Phase**:
| Phase | Turn Range | Recommended Strategy |
|-------|------------|---------------------|
| Early | 1-100 | Greedy (neutral expansion) |
| Mid   | 101-300 | Hybrid (balanced) |
| Late  | 301-500 | Aggressive (win-focused) |

In [ ]:
# ========================================
# IMPORTS AND CONFIGURATION
# ========================================

import math
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from enum import Enum

## Configuration Constants

In [ ]:
# Game Constants
BOARD_SIZE = 100.0
SUN_POSITION = (50.0, 50.0)
SUN_RADIUS = 10.0
MAX_SPEED = 6.0
MAX_TURNS = 500

# Player Colors for Visualization
PLAYER_COLORS = {
    0: '#22c55e',  # Green
    1: '#ef4444',  # Red
    2: '#3b82f6',  # Blue
    3: '#f59e0b',  # Orange
    4: '#8b5cf6',  # Purple
    5: '#ec4899',  # Pink
}

## Data Structures

In [ ]:
# ========================================
# DATA STRUCTURES
# ========================================

class GamePhase(Enum):
    """Game phases based on turn count."""
    EARLY = 1   # Turns 1-100
    MID = 2     # Turns 101-300
    LATE = 3    # Turns 301-500

@dataclass
class Planet:
    """Planet data structure."""
    id: int
    owner: int
    x: float
    y: float
    radius: float
    ships: int
    production: int
    angular_velocity: float = 0.0
    angle: float = 0.0

    def distance_to(self, other: 'Planet') -> float:
        """Calculate Euclidean distance to another planet."""
        return math.sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)

    def angle_to(self, other: 'Planet') -> float:
        """Calculate angle to another planet in radians."""
        return math.atan2(other.y - self.y, other.x - self.x)

@dataclass
class Fleet:
    """Fleet data structure."""
    id: int
    owner: int
    x: float
    y: float
    angle: float
    from_planet_id: int
    ships: int

## Game Analyzer - Core State Analysis

In [ ]:
# ========================================
# GAME ANALYZER
# ========================================

class GameAnalyzer:
    """
    Comprehensive game state analyzer.
    
    Provides utilities for understanding the current game state
    and calculating strategic values for decision making.
    """

    def __init__(self, obs: dict, player: int = 0):
        """
        Initialize analyzer with observation data.
        
        Args:
            obs: Kaggle environment observation dictionary
            player: Current player ID
        """
        self.obs = obs
        self.player = player
        self.turn = obs.get('turn', 1)
        self.angular_velocities = obs.get('angular_velocity', [])
        
        # Parse planets and fleets
        self.planets = self._parse_planets(obs.get('planets', []))
        self.fleets = self._parse_fleets(obs.get('fleets', []))
        
        # Calculate derived state
        self.phase = self._detect_game_phase()
        self.my_planets = [p for p in self.planets if p.owner == self.player]
        self.enemy_planets = [p for p in self.planets if p.owner != self.player and p.owner != -1]
        self.neutral_planets = [p for p in self.planets if p.owner == -1]
        
        # Stats
        self.my_total_ships = sum(p.ships for p in self.my_planets)
        self.my_fleet_ships = sum(f.ships for f in self.fleets if f.owner == self.player)
        self.total_my_ships = self.my_total_ships + self.my_fleet_ships

    def _parse_planets(self, planets_data: List) -> List[Planet]:
        """Parse planet data from observation."""
        planets = []
        for p in planets_data:
            planet = Planet(
                id=p[0],
                owner=p[1],
                x=p[2],
                y=p[3],
                radius=p[4],
                ships=p[5],
                production=p[6]
            )
            # Add angular velocity if available
            for av in self.angular_velocities:
                if av[0] == planet.id:
                    planet.angular_velocity = av[1]
                    planet.angle = av[2]
                    break
            planets.append(planet)
        return planets

    def _parse_fleets(self, fleets_data: List) -> List[Fleet]:
        """Parse fleet data from observation."""
        return [
            Fleet(
                id=f[0],
                owner=f[1],
                x=f[2],
                y=f[3],
                angle=f[4],
                from_planet_id=f[5],
                ships=f[6]
            )
            for f in fleets_data
        ]

    def _detect_game_phase(self) -> GamePhase:
        """Detect current game phase based on turn count."""
        if self.turn <= 100:
            return GamePhase.EARLY
        elif self.turn <= 300:
            return GamePhase.MID
        else:
            return GamePhase.LATE

    def calculate_fleet_speed(self, ships: int) -> float:
        """
        Calculate fleet speed based on ship count.
        
        Formula: speed = 1.0 + (maxSpeed - 1.0) * (log(ships) / log(1000)) ^ 1.5
        """
        if ships <= 0:
            return 1.0
        return 1.0 + (self.MAX_SPEED - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5

    def will_fleet_hit_sun(self, start_x: float, start_y: float,
                           angle: float, distance: float) -> bool:
        """
        Check if a fleet trajectory will cross the sun.
        
        Uses line-circle intersection test.
        """
        dx = math.cos(angle)
        dy = math.sin(angle)

        # Quadratic coefficients for line-circle intersection
        a = dx * dx + dy * dy
        b = 2 * (start_x * dx + start_y * dy - 50 * dx - 50 * dy)
        c = (start_x - 50) ** 2 + (start_y - 50) ** 2 - SUN_RADIUS ** 2

        discriminant = b * b - 4 * a * c
        if discriminant < 0:
            return False

        sqrt_disc = math.sqrt(discriminant)
        t1 = (-b - sqrt_disc) / (2 * a)
        t2 = (-b + sqrt_disc) / (2 * a)

        # Check if intersection occurs within travel distance
        if t1 > 0 and t1 < distance:
            return True
        if t2 > 0 and t2 < distance:
            return True

        return False

    def find_safe_launch_angle(self, source: Planet, target: Planet,
                                max_attempts: int = 36) -> Tuple[float, bool]:
        """
        Find a safe angle to launch fleet that avoids the sun.
        
        Args:
            source: Source planet
            target: Target planet
            max_attempts: Number of angle variations to try
        
        Returns:
            Tuple of (angle, found_safe)
        """
        direct_angle = source.angle_to(target)
        direct_distance = source.distance_to(target)

        # Check direct path
        if not self.will_fleet_hit_sun(source.x, source.y, direct_angle, direct_distance):
            return (direct_angle, True)

        # Try angles offset from direct path
        angle_step = 2 * math.pi / max_attempts
        max_offset = math.pi / 2  # Don't go more than 90 degrees off

        for i in range(1, max_attempts + 1):
            offset = i * angle_step
            for sign in [1, -1]:
                angle = direct_angle + sign * min(offset, max_offset)
                if not self.will_fleet_hit_sun(source.x, source.y, angle, direct_distance):
                    return (angle, True)

        # No safe angle found
        return (direct_angle, False)

    def score_target(self, planet: Planet) -> float:
        """
        Calculate strategic value score for a planet.
        
        Higher score = higher priority target.
        """
        base_value = planet.production * 10

        # Find nearest friendly planet
        nearest_friendly = None
        min_dist = float('inf')
        for fp in self.my_planets:
            dist = fp.distance_to(planet)
            if dist < min_dist:
                min_dist = dist
                nearest_friendly = fp

        if nearest_friendly:
            distance_factor = max(0, 1 - (min_dist / 100))
        else:
            distance_factor = 0.5

        # Ships factor (fewer ships = easier target)
        if planet.owner == -1:
            ship_factor = max(0, 1 - (planet.ships / 100))
        else:
            ship_factor = min(1, planet.ships / 50)

        # Phase adjustments
        if self.phase == GamePhase.EARLY:
            value = base_value * 1.5 + planet.production * 5
            value *= (1 + distance_factor)
        elif self.phase == GamePhase.MID:
            value = base_value * 1.2
            if planet.owner != -1:
                value *= 1.3
        else:  # LATE
            value = base_value
            if planet.ships < 30:
                value *= 2.0

        return value * (1 - ship_factor * 0.3)

    def get_expandable_planets(self, min_ships: int = 15) -> List[Planet]:
        """Get planets with enough ships for expansion."""
        expandable = [p for p in self.my_planets if p.ships >= min_ships]
        return sorted(expandable, key=lambda p: p.production, reverse=True)

    def is_losing(self) -> bool:
        """Check if current player is losing."""
        counts = self.get_player_ship_counts()
        if not counts:
            return False
        my_count = counts.get(self.player, 0)
        max_count = max(counts.values())
        return my_count < max_count

    def get_player_ship_counts(self) -> Dict[int, int]:
        """Get total ship counts for all players."""
        counts = {}
        for planet in self.planets:
            owner = planet.owner
            if owner == -1:
                continue
            counts[owner] = counts.get(owner, 0) + planet.ships

        for fleet in self.fleets:
            owner = fleet.owner
            if owner == -1:
                continue
            counts[owner] = counts.get(owner, 0) + fleet.ships

        return counts

    def get_aggression_ratio(self) -> float:
        """Calculate ratio of enemy ships vs friendly ships."""
        counts = self.get_player_ship_counts()
        my_ships = counts.get(self.player, 0)
        if my_ships == 0:
            return 1.0
        enemy_ships = sum(counts.get(k, 0) for k in counts if k != self.player)
        return enemy_ships / my_ships if my_ships > 0 else 0

    def recommend_strategy(self) -> str:
        """Recommend best strategy based on game state."""
        if self.phase == GamePhase.EARLY:
            if len(self.my_planets) <= 1:
                return 'greedy'
            return 'hybrid'

        elif self.phase == GamePhase.MID:
            if self.is_losing():
                if self.get_aggression_ratio() > 1.5:
                    return 'defensive'
                return 'aggressive'
            return 'hybrid'

        else:  # LATE
            if self.is_losing():
                return 'aggressive'
            return 'hybrid'

## Strategy Implementations

In [ ]:
# ========================================
# STRATEGY IMPLEMENTATIONS
# ========================================

class GreedyStrategy:
    """
    Greedy expansion strategy focusing on capturing neutrals.
    
    Best against passive opponents in early game.
    """

    def __init__(self, min_ships_for_attack: int = 15):
        self.min_ships_for_attack = min_ships_for_attack

    def get_actions(self, analyzer: GameAnalyzer) -> List[List]:
        """Generate actions for greedy strategy."""
        actions = []
        commanded_planets = set()

        # Get expandable planets sorted by production
        expandable = analyzer.get_expandable_planets(self.min_ships_for_attack)

        for source in expandable:
            if source.id in commanded_planets:
                continue

            # Find best neutral target
            best_target = None
            best_score = -1

            for neutral in analyzer.neutral_planets:
                dist = source.distance_to(neutral)
                if dist < 1:
                    continue

                # Check if path is safe (no sun collision)
                angle = source.angle_to(neutral)
                if analyzer.will_fleet_hit_sun(source.x, source.y, angle, dist):
                    continue

                # Score based on production value and distance
                score = neutral.production * 10 / dist
                if score > best_score:
                    best_score = score
                    best_target = neutral

            if best_target and best_score > 0:
                # Calculate ships to send
                ships_available = source.ships - 10  # Keep 10 as garrison
                ships_needed = best_target.ships + 1
                ships_to_send = max(ships_needed, min(ships_available, 50))

                if ships_to_send >= self.min_ships_for_attack:
                    safe_angle = analyzer.find_safe_launch_angle(source, best_target)[0]
                    actions.append([source.id, safe_angle, ships_to_send])
                    commanded_planets.add(source.id)

        return actions


class AggressiveStrategy:
    """
    Aggressive attack strategy focusing on enemy elimination.
    
    Best against defensive or expanding opponents.
    """

    def __init__(self, min_ships_for_attack: int = 20, attack_ratio: float = 0.7):
        self.min_ships_for_attack = min_ships_for_attack
        self.attack_ratio = attack_ratio

    def get_actions(self, analyzer: GameAnalyzer) -> List[List]:
        """Generate actions for aggressive strategy."""
        actions = []
        commanded = set()
        expandable = analyzer.get_expandable_planets(self.min_ships_for_attack)

        # Priority 1: Attack weakest enemy planets
        for source in expandable:
            if source.id in commanded:
                continue

            weakest_enemy = min(analyzer.enemy_planets, key=lambda p: p.ships) if analyzer.enemy_planets else None
            if not weakest_enemy:
                continue

            ships_available = source.ships - 10
            ships_needed = weakest_enemy.ships + 1

            if ships_available >= ships_needed * 1.2:  # 20% surplus for safety
                ships_to_send = int(ships_available * self.attack_ratio)
                safe_angle = analyzer.find_safe_launch_angle(source, weakest_enemy)[0]
                actions.append([source.id, safe_angle, ships_to_send])
                commanded.add(source.id)

        # Priority 2: Attack any reachable enemy
        if len(actions) < 2:
            for source in expandable:
                if source.id in commanded:
                    continue

                for enemy in analyzer.enemy_planets:
                    dist = source.distance_to(enemy)
                    if dist > 60:
                        continue

                    angle = source.angle_to(enemy)
                    if analyzer.will_fleet_hit_sun(source.x, source.y, angle, dist):
                        continue

                    ships_available = source.ships - 8
                    if ships_available >= (enemy.ships + 1) * 1.1:
                        ships_to_send = max(
                            enemy.ships + 1,
                            int(ships_available * self.attack_ratio)
                        )
                        safe_angle = analyzer.find_safe_launch_angle(source, enemy)[0]
                        actions.append([source.id, safe_angle, ships_to_send])
                        commanded.add(source.id)
                        break

        return actions


class DefensiveStrategy:
    """
    Defensive consolidation strategy.
    
    Focuses on strengthening positions and defending against attacks.
    """

    def __init__(self, garrison_target: int = 15, reinforce_threshold: int = 25):
        self.garrison_target = garrison_target
        self.reinforce_threshold = reinforce_threshold

    def get_actions(self, analyzer: GameAnalyzer) -> List[List]:
        """Generate actions for defensive strategy."""
        actions = []
        vulnerable = []
        well_defended = []

        # Identify vulnerable and well-defended planets
        for planet in analyzer.my_planets:
            if planet.ships < self.garrison_target:
                vulnerable.append(planet)
            else:
                well_defended.append(planet)

        # Priority 1: Reinforce vulnerable planets
        for vulnerable_planet in vulnerable:
            best_reinforcer = None
            max_ships = 0

            for planet in well_defended:
                if planet.ships > self.garrison_target + 10:
                    if planet.ships > max_ships:
                        max_ships = planet.ships
                        best_reinforcer = planet

            if best_reinforcer:
                ships_needed = self.garrison_target - vulnerable_planet.ships + 5
                ships_to_send = min(ships_needed, best_reinforcer.ships - self.garrison_target)

                if ships_to_send >= 8:
                    angle = best_reinforcer.angle_to(vulnerable_planet)
                    actions.append([best_reinforcer.id, angle, ships_to_send])
                    well_defended = [p for p in well_defended if p.id != best_reinforcer.id]

        # Priority 2: Capture nearby neutrals if well defended
        if len(actions) < 2:
            for planet in well_defended:
                if planet.ships < self.reinforce_threshold:
                    continue

                best_target = None
                best_score = -1

                for neutral in analyzer.neutral_planets:
                    dist = planet.distance_to(neutral)
                    if dist > 40:
                        continue

                    angle = planet.angle_to(neutral)
                    if analyzer.will_fleet_hit_sun(planet.x, planet.y, angle, dist):
                        continue

                    score = neutral.production * 10 / max(dist, 1)
                    if score > best_score:
                        best_score = score
                        best_target = neutral

                if best_target:
                    ships_available = planet.ships - self.garrison_target
                    ships_to_send = max(best_target.ships + 1, min(20, ships_available))
                    if ships_to_send >= 10:
                        safe_angle = analyzer.find_safe_launch_angle(planet, best_target)[0]
                        actions.append([planet.id, safe_angle, ships_to_send])
                        well_defended = [p for p in well_defended if p.id != planet.id]
                        break

        return actions


class HybridStrategy:
    """
    Hybrid strategy combining greedy, aggressive, and defensive approaches.
    
    Automatically adjusts priorities based on game state.
    """

    def __init__(self):
        self.greedy = GreedyStrategy(min_ships_for_attack=15)
        self.aggressive = AggressiveStrategy(min_ships_for_attack=20, attack_ratio=0.6)
        self.defensive = DefensiveStrategy(garrison_target=12, reinforce_threshold=22)

    def get_actions(self, analyzer: GameAnalyzer) -> List[List]:
        """Generate actions using hybrid approach."""
        actions = []
        action_sources = set()

        # Determine game situation
        phase = analyzer.phase
        is_losing = analyzer.is_losing()
        aggression_ratio = analyzer.get_aggression_ratio()

        # Decide primary mode
        if phase == GamePhase.EARLY:
            primary_mode = 'greedy'
        elif phase == GamePhase.MID:
            if is_losing and aggression_ratio > 1.3:
                primary_mode = 'defensive'
            elif len(analyzer.my_planets) < 2:
                primary_mode = 'greedy'
            else:
                primary_mode = 'balanced'
        else:  # LATE
            if is_losing:
                primary_mode = 'aggressive'
            else:
                primary_mode = 'balanced'

        # Execute based on primary mode
        if primary_mode == 'greedy':
            actions = self.greedy.get_actions(analyzer)
            actions.extend(self.defensive.get_actions(analyzer)[:1])
            
        elif primary_mode == 'aggressive':
            aggressive_actions = self.aggressive.get_actions(analyzer)
            actions.extend(aggressive_actions[:2])
            
            # Fill remaining capacity with greedy
            if len(actions) < 2:
                greedy_actions = self.greedy.get_actions(analyzer)
                for action in greedy_actions:
                    if len(actions) >= 3:
                        break
                    if action[0] not in action_sources:
                        actions.append(action)
                        action_sources.add(action[0])
            
        elif primary_mode == 'defensive':
            defensive_actions = self.defensive.get_actions(analyzer)
            actions.extend(defensive_actions[:2])
            
            # Check for easy kills
            easy_kills = self._find_easy_kills(analyzer)
            for kill in easy_kills:
                if len(actions) >= 3:
                    break
                if kill[0] not in action_sources:
                    actions.append(kill)
                    action_sources.add(kill[0])
            
        else:  # balanced
            # Priority 1: Easy neutral captures
            greedy_actions = self.greedy.get_actions(analyzer)
            for action in greedy_actions[:1]:
                actions.append(action)
                action_sources.add(action[0])
            
            # Priority 2: Weak enemy targets
            easy_kills = self._find_easy_kills(analyzer)
            for kill in easy_kills:
                if len(actions) >= 2:
                    break
                if kill[0] not in action_sources:
                    actions.append(kill)
                    action_sources.add(kill[0])
            
            # Priority 3: Reinforce if needed
            if len(actions) < 2:
                defensive_actions = self.defensive.get_actions(analyzer)
                for action in defensive_actions[:1]:
                    if action[0] not in action_sources:
                        actions.append(action)
                        action_sources.add(action[0])

        return actions[:3]  # Limit to 3 actions max

    def _find_easy_kills(self, analyzer: GameAnalyzer) -> List[List]:
        """Find weak enemy planets that can be easily captured."""
        easy_kills = []

        for source in analyzer.get_expandable_planets(15):
            for enemy in analyzer.enemy_planets:
                dist = source.distance_to(enemy)
                if dist > 50:
                    continue

                angle = source.angle_to(enemy)
                if analyzer.will_fleet_hit_sun(source.x, source.y, angle, dist):
                    continue

                ships_available = source.ships - 10
                ships_needed = enemy.ships + 1

                # Easy kill: we have enough ships with 20% surplus
                if ships_available >= ships_needed * 1.2:
                    ships_to_send = max(ships_needed, int(ships_available * 0.6))
                    safe_angle = analyzer.find_safe_launch_angle(source, enemy)[0]
                    easy_kills.append([source.id, safe_angle, ships_to_send])
                    break

        return easy_kills

## Main Agent - Orbit Wars Pro

In [ ]:
# ========================================
# MAIN AGENT - ORBIT WARS PRO
# ========================================

class OrbitWarsPro:
    """
    Professional Orbit Wars agent with multi-strategy support.
    
    Features:
    - Multiple built-in strategies (greedy, aggressive, defensive, hybrid)
    - Automatic strategy selection based on game state
    - Configurable strategy mode
    """

    STRATEGY_GREEDY = 'greedy'
    STRATEGY_AGGRESSIVE = 'aggressive'
    STRATEGY_DEFENSIVE = 'defensive'
    STRATEGY_HYBRID = 'hybrid'
    STRATEGY_ADAPTIVE = 'adaptive'

    def __init__(self, strategy_mode: str = STRATEGY_ADAPTIVE, debug: bool = False):
        """
        Initialize Orbit Wars Pro agent.
        
        Args:
            strategy_mode: Strategy to use ('greedy', 'aggressive', 'defensive', 'hybrid', 'adaptive')
            debug: Enable debug output
        """
        self.strategy_mode = strategy_mode
        self.debug = debug

        # Initialize strategies
        self.strategies = {
            self.STRATEGY_GREEDY: GreedyStrategy(min_ships_for_attack=15),
            self.STRATEGY_AGGRESSIVE: AggressiveStrategy(min_ships_for_attack=18, attack_ratio=0.65),
            self.STRATEGY_DEFENSIVE: DefensiveStrategy(garrison_target=12, reinforce_threshold=22),
            self.STRATEGY_HYBRID: HybridStrategy(),
        }

        # Track state
        self.current_strategy = strategy_mode
        self.turn_count = 0

    def set_strategy(self, mode: str):
        """Set the active strategy mode."""
        if mode in self.strategies or mode == self.STRATEGY_ADAPTIVE:
            self.strategy_mode = mode
            self.current_strategy = mode

    def act(self, obs: dict) -> List[List]:
        """
        Generate actions based on observation.
        
        Args:
            obs: Kaggle environment observation
        
        Returns:
            List of fleet commands [[from_id, angle, ships], ...]
        """
        # Update turn count
        self.turn_count = obs.get('turn', self.turn_count + 1)

        # Analyze game state
        analyzer = GameAnalyzer(obs, obs.get('player', 0))

        # Log debug info
        if self.debug:
            print(f"\n=== Orbit Wars Pro Debug (Turn {analyzer.turn}) ===")
            print(f"Phase: {analyzer.phase.name}")
            print(f"My planets: {len(analyzer.my_planets)}")
            print(f"Enemy planets: {len(analyzer.enemy_planets)}")
            print(f"Neutral planets: {len(analyzer.neutral_planets)}")
            print(f"My total ships: {analyzer.my_total_ships}")
            print(f"Recommended strategy: {analyzer.recommend_strategy()}")

        # Determine which strategy to use
        if self.strategy_mode == self.STRATEGY_ADAPTIVE:
            self.current_strategy = analyzer.recommend_strategy()
        else:
            self.current_strategy = self.strategy_mode

        # Execute strategy
        strategy = self.strategies.get(
            self.current_strategy,
            self.strategies[self.STRATEGY_HYBRID]
        )

        actions = strategy.get_actions(analyzer)

        # Log actions
        if self.debug and actions:
            print(f"Strategy: {self.current_strategy}")
            print(f"Actions: {actions}")

        return actions

    def get_stats(self, obs: dict) -> Dict:
        """
        Get agent statistics for analysis.
        
        Args:
            obs: Current observation
        
        Returns:
            Dictionary with agent stats
        """
        analyzer = GameAnalyzer(obs, obs.get('player', 0))

        return {
            'turn': analyzer.turn,
            'phase': analyzer.phase.name,
            'my_planets': len(analyzer.my_planets),
            'enemy_planets': len(analyzer.enemy_planets),
            'neutral_planets': len(analyzer.neutral_planets),
            'my_ships': analyzer.my_total_ships,
            'is_losing': analyzer.is_losing(),
            'recommended_strategy': analyzer.recommend_strategy(),
            'current_strategy': self.current_strategy,
        }

## Kaggle Agent Entry Point

In [ ]:
# ========================================
# KAGGLE ENTRY POINT
# ========================================

# Global instance for simple usage
_agent_instance = None

def orbit_wars_agent(obs: dict) -> List[List]:
    """
    Main agent function for Kaggle submission.
    
    This is the entry point for the Kaggle environment.
    
    Args:
        obs: Kaggle environment observation
    
    Returns:
        List of fleet commands
    """
    global _agent_instance

    # Initialize or get agent instance
    if _agent_instance is None:
        _agent_instance = OrbitWarsPro(strategy_mode='adaptive', debug=False)

    return _agent_instance.act(obs)


# Alternative entry points for specific strategies
def greedy_agent(obs: dict) -> List[List]:
    """Greedy strategy agent."""
    analyzer = GameAnalyzer(obs, obs.get('player', 0))
    strategy = GreedyStrategy(min_ships_for_attack=15)
    return strategy.get_actions(analyzer)


def aggressive_agent(obs: dict) -> List[List]:
    """Aggressive strategy agent."""
    analyzer = GameAnalyzer(obs, obs.get('player', 0))
    strategy = AggressiveStrategy(min_ships_for_attack=18, attack_ratio=0.65)
    return strategy.get_actions(analyzer)


def defensive_agent(obs: dict) -> List[List]:
    """Defensive strategy agent."""
    analyzer = GameAnalyzer(obs, obs.get('player', 0))
    strategy = DefensiveStrategy(garrison_target=12, reinforce_threshold=22)
    return strategy.get_actions(analyzer)


def hybrid_agent(obs: dict) -> List[List]:
    """Hybrid strategy agent."""
    analyzer = GameAnalyzer(obs, obs.get('player', 0))
    strategy = HybridStrategy()
    return strategy.get_actions(analyzer)

## Testing and Validation

In [ ]:
# ========================================
# TESTING AND VALIDATION
# ========================================

def test_agent():
    """Test the agent with sample observations."""
    
    # Test 1: Early game scenario
    early_game_obs = {
        'turn': 50,
        'player': 0,
        'planets': [
            [0, 0, 25, 25, 2.5, 30, 3],  # My home planet
            [1, -1, 40, 35, 2.0, 15, 2],  # Neutral
            [2, -1, 55, 45, 2.5, 25, 4],  # Neutral high production
            [3, 1, 70, 60, 2.5, 20, 3],    # Enemy
        ],
        'fleets': [],
        'angular_velocity': []
    }

    print("=" * 50)
    print("TEST 1: Early Game")
    print("=" * 50)
    
    agent = OrbitWarsPro(strategy_mode='adaptive', debug=True)
    actions = agent.act(early_game_obs)
    stats = agent.get_stats(early_game_obs)
    
    print(f"\nActions: {actions}")
    print(f"Stats: {stats}")
    print()

    # Test 2: Mid game scenario
    mid_game_obs = {
        'turn': 200,
        'player': 0,
        'planets': [
            [0, 0, 25, 25, 2.5, 45, 3],
            [1, 0, 40, 35, 2.0, 32, 2],
            [2, -1, 55, 45, 2.5, 18, 4],
            [3, 1, 70, 60, 2.5, 38, 3],
            [4, 0, 30, 70, 2.0, 25, 2],
            [5, 1, 75, 25, 1.8, 20, 1],
        ],
        'fleets': [
            [0, 0, 35, 32, 0.5, 0, 15],
            [1, 1, 60, 50, 3.5, 3, 18],
        ],
        'angular_velocity': []
    }

    print("=" * 50)
    print("TEST 2: Mid Game")
    print("=" * 50)
    
    agent = OrbitWarsPro(strategy_mode='adaptive', debug=True)
    actions = agent.act(mid_game_obs)
    stats = agent.get_stats(mid_game_obs)
    
    print(f"\nActions: {actions}")
    print(f"Stats: {stats}")
    print()

    # Test 3: Late game scenario
    late_game_obs = {
        'turn': 400,
        'player': 0,
        'planets': [
            [0, 0, 25, 25, 2.5, 55, 3],
            [1, 0, 40, 35, 2.0, 42, 2],
            [2, 0, 55, 45, 2.5, 35, 4],
            [3, 1, 70, 60, 2.5, 48, 3],
            [4, 1, 30, 70, 2.0, 30, 2],
            [5, 0, 75, 25, 1.8, 25, 1],
        ],
        'fleets': [
            [0, 0, 35, 32, 0.5, 0, 20],
            [1, 0, 60, 55, 3.8, 2, 25],
            [2, 1, 50, 45, 2.5, 3, 18],
        ],
        'angular_velocity': []
    }

    print("=" * 50)
    print("TEST 3: Late Game")
    print("=" * 50)
    
    agent = OrbitWarsPro(strategy_mode='adaptive', debug=True)
    actions = agent.act(late_game_obs)
    stats = agent.get_stats(late_game_obs)
    
    print(f"\nActions: {actions}")
    print(f"Stats: {stats}")
    print()

    # Test 4: Different strategies
    print("=" * 50)
    print("TEST 4: Different Strategies")
    print("=" * 50)
    
    for strategy_name in ['greedy', 'aggressive', 'defensive', 'hybrid']:
        agent = OrbitWarsPro(strategy_mode=strategy_name, debug=False)
        actions = agent.act(early_game_obs)
        print(f"{strategy_name.capitalize()}: {actions}")

    print()
    print("All tests completed successfully!")


# Run tests
test_agent()

## Strategy Performance Analysis

In [ ]:
# ========================================
# STRATEGY PERFORMANCE ANALYSIS
# ========================================

import random

def generate_random_observation():
    """Generate a random game observation for testing."""
    turn = random.randint(1, 500)
    phase = 'EARLY' if turn <= 100 else 'MID' if turn <= 300 else 'LATE'
    
    planets = []
    
    # Player 0 home planet
    planets.append([0, 0, random.uniform(15, 35), random.uniform(15, 35), 2.5, random.randint(20, 50), 3])
    
    # Player 1 home planet (opposite side)
    planets.append([1, 1, random.uniform(65, 85), random.uniform(65, 85), 2.5, random.randint(20, 50), 3])
    
    # Neutral planets
    for i in range(2, random.randint(4, 8)):
        x = random.uniform(20, 80)
        y = random.uniform(20, 80)
        production = random.randint(1, 5)
        radius = 1 + math.log(production)
        ships = random.randint(5, 40)
        planets.append([i, -1, x, y, radius, ships, production])
    
    return {
        'turn': turn,
        'player': 0,
        'planets': planets,
        'fleets': [],
        'angular_velocity': []
    }


def analyze_strategies(num_tests: int = 20):
    """Analyze strategy performance across multiple random scenarios."""
    
    strategies = {
        'greedy': lambda obs: GreedyStrategy().get_actions(GameAnalyzer(obs, 0)),
        'aggressive': lambda obs: AggressiveStrategy().get_actions(GameAnalyzer(obs, 0)),
        'defensive': lambda obs: DefensiveStrategy().get_actions(GameAnalyzer(obs, 0)),
        'hybrid': lambda obs: HybridStrategy().get_actions(GameAnalyzer(obs, 0)),
    }
    
    results = {name: {'actions': 0, 'total_ships': 0} for name in strategies.keys()}
    
    print("Analyzing strategies across random scenarios...\n")
    
    for i in range(num_tests):
        obs = generate_random_observation()
        
        for name, strategy_fn in strategies.items():
            try:
                actions = strategy_fn(obs)
                ships_sent = sum(a[2] for a in actions)
                results[name]['actions'] += len(actions)
                results[name]['total_ships'] += ships_sent
            except Exception as e:
                print(f"Error in {name}: {e}")
    
    print("Strategy Performance Summary:")
    print("-" * 40)
    for name, data in results.items():
        avg_actions = data['actions'] / num_tests if num_tests > 0 else 0
        avg_ships = data['total_ships'] / num_tests if num_tests > 0 else 0
        print(f"{name.capitalize():12}: Avg {avg_actions:.2f} actions, {avg_ships:.1f} ships sent")


analyze_strategies(20)

## Usage Instructions for Kaggle

### Option 1: Use the Main Agent (Recommended)

```python
from orbit_wars_agent import orbit_wars_agent

def your_agent(obs):
    return orbit_wars_agent(obs)
```

### Option 2: Use Specific Strategies

```python
from orbit_wars_agent import greedy_agent, aggressive_agent, defensive_agent, hybrid_agent

def your_agent(obs):
    return hybrid_agent(obs)  # Choose your strategy
```

### Option 3: Customize the Agent

```python
from orbit_wars_agent import OrbitWarsPro

agent = OrbitWarsPro(strategy_mode='adaptive')  # or 'greedy', 'aggressive', etc.

def your_agent(obs):
    return agent.act(obs)
```

---

## Summary

This notebook provides a complete, professional-grade solution for the Orbit Wars Kaggle competition.

### Key Features:

| Feature | Description |
|---------|-------------|
| **5 Strategy Modes** | Adaptive, Greedy, Aggressive, Defensive, Hybrid |
| **Auto-Selection** | Automatically chooses best strategy based on game state |
| **Sun Avoidance** | Calculates safe trajectories avoiding the sun |
| **Phase Detection** | Early/Mid/Late game logic |
| **Threat Assessment** | Evaluates enemy aggression levels |
| **Fleet Optimization** | Calculates optimal ship counts for attacks |

### Competition Details:

- **Prize Pool**: $50,000
- **Entry Deadline**: June 16, 2026
- **Final Submission**: June 23, 2026
- **Evaluation Period**: June 24 - ~July 8, 2026

### Tips for Improvement:

1. **Tune strategy thresholds** in each strategy class
2. **Consider orbital mechanics** - some planets rotate around the sun
3. **Monitor fleet speed** - larger fleets travel faster
4. **Watch for comets** - temporary planets that spawn periodically
5. **Balance offense/defense** based on your position

**Good luck in the competition!** 🏆